# ADVC — Kaggle Notebook (Tiny-ImageNet, T4 GPU)

**Environment:** Kaggle Notebooks · GPU T4 x2 (16 GB each) · ~4 CPU cores · ~30 GB RAM

Runs top-to-bottom in **batch mode** (Save & Run All) so it keeps going with your PC off. Every phase is **resumable** — recorded CSV rows are skipped and trained levels reload via `--skip-training`.

### The one cell you edit each run: **Cell 5d — Batch Control Panel**
Because Save & Run All executes every cell, you don't comment cells out. Instead **Cell 5d** has switches (`RUN_PHASE1`, `P2A_LEVELS`, `P2B_LEVELS`, `RUN_PHASE3`, `RUN_FIGURES`); each phase cell no-ops if not selected. Scope each batch to a chunk that fits ~12h — see the suggested A–E chunks in Cell 5d and the guide at the bottom.

### Before running — notebook Settings (right sidebar)
1. **Accelerator** → `GPU T4 x2`  (never run on CPU)
2. **Internet** → `On` (needed for `git clone`, `pip install`, HF model download)
3. **Add-ons → Secrets** → add `HF_TOKEN` = your HuggingFace token
4. **Add-ons → Datasets** → attach a Tiny-ImageNet dataset, AND (from batch 2 on) your saved `advc-results` dataset so Cell 5b restores prior work.

### Persistence
`/kaggle/working` is wiped between sessions. Cell 13 saves `results/` (CSVs + checkpoints) to the `advc-results` Kaggle dataset at the end of every run; attach it next batch and Cell 5b restores it. Set `KAGGLE_USER` in Cell 5d once so the first save can create the dataset.

In [ ]:
# Cell 1 — Verify GPU
import torch

print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free, total = torch.cuda.mem_get_info(0)
    print('GPU name       :', name)
    print(f'Free VRAM      : {free/1e9:.1f} GB / {total/1e9:.1f} GB total')
    if 'T4' not in name:
        print('\nWARNING: expected Tesla T4 — check Settings > Accelerator > GPU T4 x2')
    else:
        print('\nT4 confirmed. Ready to proceed.')
else:
    print('\nWARNING: no GPU. Set Settings > Accelerator > GPU T4 x2, then restart.')

In [ ]:
# Cell 2 — HF token + dependencies
import os, subprocess, sys

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets.')
except Exception as e:
    print('WARNING: could not load HF_TOKEN secret:', e)
    print('Add it in Settings > Add-ons > Secrets, or downloads may be rate-limited.')

packages = ['timm', 'torchattacks', 'bitsandbytes', 'optimum', 'pyyaml',
            'tqdm', 'accelerate', 'huggingface_hub', 'transformers>=4.44.0,<5.0']
res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages,
                     capture_output=True, text=True)
if res.returncode != 0:
    print('pip error:\n', res.stderr[-2000:])
else:
    print('Installed:', ', '.join(packages))

In [ ]:
# Cell 3 — Clone / update repo, set cwd
import os, sys, subprocess

REPO_URL = 'https://github.com/Jmanav/ADVC.git'
REPO_DIR = '/kaggle/working/ADVC'

if not os.path.isdir(REPO_DIR):
    print('Cloning', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo exists — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('cwd is now:', os.getcwd())
subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-1'])
assert os.path.exists('configs/base.yaml'), 'configs/base.yaml missing — clone failed?'

In [ ]:
# Cell 4 — Extract + prepare Tiny-ImageNet
# Source may be read-only (/kaggle/input mount). Output MUST be writable
# (/kaggle/working), so --root and --out are kept separate.
import os, glob, zipfile, subprocess, sys

SRC_ROOT = None
OUT_ROOT = '/kaggle/working/tiny_if'   # WRITABLE — matches configs/base.yaml

for p in glob.glob('/kaggle/input/**/tiny-imagenet-200', recursive=True):
    if os.path.isdir(p):
        SRC_ROOT = p
        break

if SRC_ROOT is None:
    zips = (glob.glob('/kaggle/input/**/tiny-imagenet-200.zip', recursive=True)
            or glob.glob('/kaggle/input/**/*tiny*imagenet*.zip', recursive=True))
    assert zips, ('No tiny-imagenet-200 folder or zip under /kaggle/input. '
                  'Attach a Tiny-ImageNet dataset in Settings > Add-ons > Datasets.')
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall('/kaggle/working/')
    SRC_ROOT = '/kaggle/working/tiny-imagenet-200'

print('Source:', SRC_ROOT, '\nOutput:', OUT_ROOT)

res = subprocess.run(
    [sys.executable, 'scripts/prepare_tiny_imagenet.py',
     '--root', SRC_ROOT, '--out', OUT_ROOT],
    capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print('prepare failed:\n', res.stderr[-2000:])
else:
    for d in ['train_if', 'val_if']:
        path = os.path.join(OUT_ROOT, d)
        n = len(os.listdir(path)) if os.path.isdir(path) else 0
        print(f'  {d}: {n} class folders')
    print('Tiny-ImageNet ready.')

In [ ]:
# Cell 5 — Output directories (dataset-scoped, matching the code paths)
import os
DS_NAME = 'tiny-imagenet'
for d in ['results', f'results/{DS_NAME}',
          f'results/checkpoints/{DS_NAME}/at', f'results/checkpoints/{DS_NAME}/atkd',
          f'results/{DS_NAME}/figures']:
    os.makedirs(d, exist_ok=True)
    print('Ready:', d)

## Cell 5b — Restore prior results (CSVs + checkpoints)

From session 2 onward, attach your saved `advc-results` dataset (Add-ons > Datasets). This cell auto-finds it under `/kaggle/input` and restores files to the **exact paths the scripts read**:
- CSVs → `results/tiny-imagenet/` (per-dataset subfolder — this is where `dataset_results_path` writes them)
- Checkpoints → `results/checkpoints/{at,atkd}/`

If nothing is attached it prints a clear warning and everything computes fresh.

In [ ]:
# Cell 5b — Restore prior results (CSVs + checkpoints) from a saved dataset
# Checkpoints and figures are now dataset-scoped in code, so restore preserves
# that structure: results/checkpoints/<DS_NAME>/{at,atkd}/ and results/<DS_NAME>/.
import os, glob, shutil

DS_NAME = 'tiny-imagenet'  # cfg['dataset']['name']
os.makedirs(f'results/{DS_NAME}', exist_ok=True)
os.makedirs(f'results/checkpoints/{DS_NAME}/at', exist_ok=True)
os.makedirs(f'results/checkpoints/{DS_NAME}/atkd', exist_ok=True)

def _find(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None

# 1) CSVs -> results/<DS_NAME>/   (match only this dataset's subfolder to avoid mixing)
restored_csv = 0
for csv_name in ['phase1_results.csv', 'phase2_at_results.csv',
                 'phase2_atkd_results.csv', 'phase3_results.csv']:
    src = _find(f'/kaggle/input/**/{DS_NAME}/{csv_name}')
    if src:
        shutil.copy(src, f'results/{DS_NAME}/{csv_name}')
        print('Restored CSV        ->', src)
        restored_csv += 1

# 2) Checkpoints -> results/checkpoints/<DS_NAME>/{at,atkd}/  (dataset-scoped source)
restored_ck = 0
for sub in ['at', 'atkd']:
    for ckpt in glob.glob(f'/kaggle/input/**/checkpoints/{DS_NAME}/{sub}/*.pt', recursive=True):
        shutil.copy(ckpt, f'results/checkpoints/{DS_NAME}/{sub}/{os.path.basename(ckpt)}')
        restored_ck += 1
    n = len(os.listdir(f'results/checkpoints/{DS_NAME}/{sub}'))
    print(f'checkpoints/{DS_NAME}/{sub}: {n} file(s) present')

if restored_csv == 0 and restored_ck == 0:
    print('\nWARNING: nothing restored from /kaggle/input for dataset', DS_NAME, '. '
          'If this is not your first session, attach your advc-results dataset — '
          'otherwise everything will recompute from scratch.')

In [ ]:
# Cell 5c — Verify resume state (run before any training cell)
# Answers "is it going to restart from the beginning?" BEFORE you spend GPU.
import os, glob, pandas as pd

DS_NAME = 'tiny-imagenet'
EPOCHS = 7  # cfg['defense']['epochs'] — --skip-training only reuses the FINAL epoch

print('=== CSV rows already recorded (these attack rows will be SKIPPED) ===')
for name in ['phase1_results.csv', 'phase2_at_results.csv',
             'phase2_atkd_results.csv', 'phase3_results.csv']:
    p = f'results/{DS_NAME}/{name}'
    if os.path.exists(p):
        df = pd.read_csv(p)
        combos = sorted(set(zip(df.get('compression', []), df.get('attack', []))))
        print(f'  {name}: {len(df)} rows -> {combos}')
    else:
        print(f'  {name}: MISSING (will compute fresh)')

print('\n=== Reusable checkpoints (epoch{:02d} = loadable via --skip-training) ==='.format(EPOCHS))
for sub, prefix in [('at', 'at'), ('atkd', 'atkd')]:
    for lvl in ['fp32', 'int8', 'int4']:
        hit = glob.glob(f'results/checkpoints/{DS_NAME}/{sub}/{prefix}_{lvl}_epoch{EPOCHS:02d}*.pt')
        status = ('REUSABLE ' + os.path.basename(hit[0])) if hit else 'none -> will TRAIN'
        print(f'  {sub}/{lvl}: {status}')

In [ ]:
# Cell 5d — BATCH CONTROL PANEL  (edit this ONE cell, then Save & Run All)
# ---------------------------------------------------------------------------
# In batch mode (Save & Run All) every cell runs top-to-bottom, so instead of
# commenting cells out, each phase cell checks these switches and no-ops if the
# phase/level is not selected for THIS run. Pick a chunk that fits in ~12h.
#
# Suggested chunks (each fits one Kaggle batch):
#   A: RUN_PHASE1=True,  P2A_LEVELS=['fp32','int8'], rest empty        (~6h)
#   B: P2A_LEVELS=['int4']                                            (~3h)
#   C: P2B_LEVELS=['fp32','int8']                                     (~7h)
#   D: P2B_LEVELS=['int4']                                            (~4h)
#   E: RUN_PHASE3=True, RUN_FIGURES=True                              (~4h)
# ---------------------------------------------------------------------------
RUN_PHASE1  = True                 # Cell 7  — no-defense baseline (fast, ~2h)
P2A_LEVELS  = ['fp32', 'int8']     # Cell 8  — AT levels to run this batch ([] = skip)
P2B_LEVELS  = []                   # Cell 9  — AT+KD levels to run this batch ([] = skip)
RUN_PHASE3  = False                # Cell 10 — combined attack (needs Phase 2 checkpoints)
RUN_FIGURES = False                # Cell 12 — paper figures (needs all CSVs)

# Persistence: save results to a Kaggle dataset at the end so the NEXT batch resumes.
SAVE_DATASET   = True
DATASET_SLUG   = 'advc-results'    # -> <your-username>/advc-results
KAGGLE_USER    = ''                # REQUIRED for the first-ever save; e.g. 'jmanav'

print('This batch will run:')
print('  Phase 1     :', RUN_PHASE1)
print('  Phase 2a AT :', P2A_LEVELS or 'skip')
print('  Phase 2b KD :', P2B_LEVELS or 'skip')
print('  Phase 3     :', RUN_PHASE3)
print('  Figures     :', RUN_FIGURES)
print('  Save dataset:', SAVE_DATASET, f'({KAGGLE_USER}/{DATASET_SLUG})' if KAGGLE_USER else '(user not set)')

In [ ]:
# Cell 6 — Smoke test
import torch
from models.loader import load_config, load_model

cfg = load_config('configs/base.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dataset in config:', cfg['dataset']['name'])

model = load_model('deit_small', 'fp32', cfg, device=device)
model.eval()
dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
    if hasattr(out, 'logits'):
        out = out.logits
print('Output shape:', tuple(out.shape), '(expected (2, 1000))')

del model, dummy
torch.cuda.empty_cache()
print('Smoke test passed.')

# Cell 7 — Phase 1
import subprocess, sys, os

if not RUN_PHASE1:
    print('Phase 1 not selected in Cell 5d — skipping.')
else:
    proc = subprocess.Popen([sys.executable, 'experiments/eval_phase1.py', '--model', 'deit_small'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('\n[Phase 1] exit code', proc.returncode)

In [ ]:
# Cell 7 — Phase 1
import subprocess, sys, os

proc = subprocess.Popen([sys.executable, 'experiments/eval_phase1.py', '--model', 'deit_small'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        bufsize=1, env={**os.environ})
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n[Phase 1] exit code', proc.returncode)

# Cell 8 — Phase 2a (AT), auto-skip already-trained levels
import subprocess, sys, os, glob

DS_NAME = 'tiny-imagenet'
EPOCHS = 7

if not P2A_LEVELS:
    print('Phase 2a: no levels selected in Cell 5d (P2A_LEVELS=[]) — skipping.')
else:
    for compression in P2A_LEVELS:
        ckpt = glob.glob(f'results/checkpoints/{DS_NAME}/at/at_{compression}_epoch{EPOCHS:02d}*.pt')
        cmd = [sys.executable, 'experiments/eval_phase2_at.py', '--compression', compression]
        if ckpt:
            cmd.append('--skip-training')
            note = f'checkpoint found ({os.path.basename(ckpt[0])}) -> --skip-training (eval only)'
        else:
            note = 'no checkpoint -> training 7 epochs'
        print('=' * 60)
        print(f'Phase 2a: AT — {compression}  [{note}]')
        print('=' * 60)
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1, env={**os.environ})
        for line in proc.stdout:
            print(line, end='', flush=True)
        proc.wait()
        print(f'[Phase 2a {compression}] exit code', proc.returncode, '\n')

In [ ]:
# Cell 8 — Phase 2a (AT), auto-skip already-trained levels
import subprocess, sys, os, glob

DS_NAME = 'tiny-imagenet'
LEVELS = ['fp32', 'int8', 'int4']   # trim to fit the 12h session cap
EPOCHS = 7

for compression in LEVELS:
    ckpt = glob.glob(f'results/checkpoints/{DS_NAME}/at/at_{compression}_epoch{EPOCHS:02d}*.pt')
    cmd = [sys.executable, 'experiments/eval_phase2_at.py', '--compression', compression]
    if ckpt:
        cmd.append('--skip-training')
        note = f'checkpoint found ({os.path.basename(ckpt[0])}) -> --skip-training (eval only)'
    else:
        note = 'no checkpoint -> training 7 epochs'
    print('=' * 60)
    print(f'Phase 2a: AT — {compression}  [{note}]')
    print('=' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print(f'[Phase 2a {compression}] exit code', proc.returncode, '\n')

# Cell 9 — Phase 2b (AT+KD), auto-skip already-trained levels
import subprocess, sys, os, glob

DS_NAME = 'tiny-imagenet'
EPOCHS = 7

if not P2B_LEVELS:
    print('Phase 2b: no levels selected in Cell 5d (P2B_LEVELS=[]) — skipping.')
else:
    for compression in P2B_LEVELS:
        ckpt = glob.glob(f'results/checkpoints/{DS_NAME}/atkd/atkd_{compression}_epoch{EPOCHS:02d}*.pt')
        cmd = [sys.executable, 'experiments/eval_phase2_atkd.py', '--compression', compression]
        if ckpt:
            cmd.append('--skip-training')
            note = f'checkpoint found ({os.path.basename(ckpt[0])}) -> --skip-training (eval only)'
        else:
            note = 'no checkpoint -> training 7 epochs'
        print('=' * 60)
        print(f'Phase 2b: AT+KD — {compression}  [{note}]')
        print('=' * 60)
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1, env={**os.environ})
        for line in proc.stdout:
            print(line, end='', flush=True)
        proc.wait()
        print(f'[Phase 2b {compression}] exit code', proc.returncode, '\n')

In [ ]:
# Cell 9 — Phase 2b (AT+KD), auto-skip already-trained levels
import subprocess, sys, os, glob

DS_NAME = 'tiny-imagenet'
LEVELS = ['fp32', 'int8', 'int4']   # trim to fit the 12h session cap
EPOCHS = 7

for compression in LEVELS:
    ckpt = glob.glob(f'results/checkpoints/{DS_NAME}/atkd/atkd_{compression}_epoch{EPOCHS:02d}*.pt')
    cmd = [sys.executable, 'experiments/eval_phase2_atkd.py', '--compression', compression]
    if ckpt:
        cmd.append('--skip-training')
        note = f'checkpoint found ({os.path.basename(ckpt[0])}) -> --skip-training (eval only)'
    else:
        note = 'no checkpoint -> training 7 epochs'
    print('=' * 60)
    print(f'Phase 2b: AT+KD — {compression}  [{note}]')
    print('=' * 60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print(f'[Phase 2b {compression}] exit code', proc.returncode, '\n')

# Cell 10 — Phase 3
import subprocess, sys, os

if not RUN_PHASE3:
    print('Phase 3 not selected in Cell 5d — skipping.')
else:
    proc = subprocess.Popen([sys.executable, 'experiments/eval_phase3.py', '--model', 'deit_small'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('\n[Phase 3] exit code', proc.returncode)

In [ ]:
# Cell 10 — Phase 3
import subprocess, sys, os

proc = subprocess.Popen([sys.executable, 'experiments/eval_phase3.py', '--model', 'deit_small'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        bufsize=1, env={**os.environ})
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n[Phase 3] exit code', proc.returncode)

In [ ]:
# Cell 12 — Paper figures
import subprocess, sys, os

DS_NAME = 'tiny-imagenet'
if not RUN_FIGURES:
    print('Figures not selected in Cell 5d — skipping.')
else:
    proc = subprocess.Popen([sys.executable, 'utils/paper_figures.py', '--n-samples', '4', '--n-eval', '200'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print('\n[Figures] exit code', proc.returncode)
    figs = os.path.join('results', DS_NAME, 'figures')  # dataset-scoped
    if os.path.isdir(figs):
        for f in sorted(os.listdir(figs)):
            if f != '.gitkeep':
                print(' ', f, f'{os.path.getsize(os.path.join(figs, f))/1024:.0f} KB')

## Cell 13 — Persist results + checkpoints (runs automatically at the end)

`/kaggle/working` is wiped between sessions, so the next cell saves `results/` (CSVs + checkpoints) to a Kaggle dataset. It **auto-creates** the dataset on the first run (needs `KAGGLE_USER` set in Cell 5d) and **versions** it on every run after. Failures here are caught and never fail the committed batch — results always remain in the version's **Output** tab.

Next session, attach this `advc-results` dataset and Cell 5b restores everything.

# Cell 13 — Persist results to a Kaggle dataset (auto create-or-version, batch-safe)
# Runs unconditionally at the end of the notebook. In batch mode it auto-creates
# the dataset on the first run and versions it afterwards, so the NEXT batch can
# resume. Any failure here is caught and reported — it never fails the committed
# run (results still live in this version's Output tab regardless).
import os, json, subprocess

RESULTS_DIR = '/kaggle/working/ADVC/results'

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr.strip()[-1500:])
    return r.returncode

if not SAVE_DATASET:
    print('SAVE_DATASET=False in Cell 5d — not persisting. (Outputs still in this version Output tab.)')
else:
    try:
        meta_path = os.path.join(RESULTS_DIR, 'dataset-metadata.json')
        exists = os.path.exists(meta_path)
        if not exists:
            if not KAGGLE_USER:
                raise RuntimeError(
                    'First save needs KAGGLE_USER set in Cell 5d (e.g. "jmanav") to create '
                    f'{DATASET_SLUG}. Set it and re-run, or create the dataset once manually.')
            json.dump({'title': DATASET_SLUG,
                       'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
                       'licenses': [{'name': 'CC0-1.0'}]},
                      open(meta_path, 'w'))
            print(f'Creating dataset {KAGGLE_USER}/{DATASET_SLUG} ...')
            rc = _run(['kaggle', 'datasets', 'create', '-p', RESULTS_DIR, '-r', 'zip'])
        else:
            print('Versioning existing dataset ...')
            rc = _run(['kaggle', 'datasets', 'version', '-p', RESULTS_DIR,
                       '-m', 'session results update', '-r', 'zip'])
        if rc == 0:
            print('\nSaved OK. Attach this dataset next session so Cell 5b restores it.')
        else:
            print('\nWARNING: save command failed (see above). Download from the Output tab '
                  'manually, or fix kaggle auth. The run itself is fine.')
    except Exception as e:
        print('WARNING: persistence step skipped:', e)
        print('Results are still in this version Output tab — download them from there.')

---
## Running with your PC off — batch mode (Save & Run All)

The full 36-row matrix (~20 GPU-h) does **not** fit in one 12h Kaggle session, so run it in chunks. Batch mode ("Save & Run All / Commit") runs the whole notebook on Kaggle's servers with your PC off — but it **re-runs every cell from scratch on a fresh machine** and **commits nothing if it times out**. So each batch must (a) be scoped to fit ~12h and (b) restore prior work from your saved dataset.

The **Cell 5d control panel** is how you scope a batch without commenting cells out — every phase cell reads it and no-ops if not selected.

### Per-batch procedure
1. **Edit only Cell 5d** to select this batch's chunk (see the suggested A–E chunks in that cell). Set `KAGGLE_USER` once for the first save.
2. **Attach datasets** (Settings → Add-ons → Datasets): Tiny-ImageNet **and** your `advc-results` (from batch 2 on). The second is what makes it resume instead of restart.
3. Confirm **GPU T4 x2**, **Internet On**, **HF_TOKEN** secret.
4. **Save Version → "Save & Run All (Commit)" → Save.** Wait ~1 min until it shows *Running*, then close the browser / shut down your PC.
5. When it finishes: notebook **Versions** tab → confirm *Complete*. Cell 13 has saved a new `advc-results` version, ready to attach for the next batch.

### Suggested chunks (one per batch)
| Batch | Cell 5d settings | Approx |
|---|---|---|
| A | `RUN_PHASE1=True`, `P2A_LEVELS=['fp32','int8']`, rest off | ~6h |
| B | `P2A_LEVELS=['int4']`, rest off | ~3h |
| C | `P2B_LEVELS=['fp32','int8']`, rest off | ~7h |
| D | `P2B_LEVELS=['int4']`, rest off | ~4h |
| E | `RUN_PHASE3=True`, `RUN_FIGURES=True`, rest off | ~4h |

### Guardrails
- **Never scope a batch beyond ~10h of work** — a timeout commits nothing.
- **Always attach `advc-results` from batch 2 on** — forget it and Cell 5b's WARNING fires and everything retrains.
- **Edits only take effect when you commit** — change Cell 5d, then Save Version; don't edit after clicking commit.
- **Kaggle has no progressive-chain scheduler on free tier** — you must edit Cell 5d + attach the dataset + click Save Version for each batch. It's PC-off per run, not hands-off across all five.

### How to confirm it is NOT restarting from scratch
In the completed batch's log, resumed levels show `checkpoint found -> --skip-training (eval only)` and `... checkpoint loaded.` with **no** `Epoch 1/7`; already-recorded attack rows show `Resuming — N combination(s) already done`. Run Cell 5c early in the log to see the reusable-checkpoint / recorded-row summary before any GPU is spent.

---
## Running across multiple sessions with your PC off

The full 36-row matrix (~20 GPU-h) does **not** fit in one 12h Kaggle session. Split it:

| Session | Cells to run | Approx |
|---|---|---|
| 1 | 1–7 (setup + Phase 1) then Cell 8 with `LEVELS=['fp32','int8']` | ~6h |
| 2 | 1–5c (restore) then Cell 8 with `LEVELS=['int4']` | ~3h |
| 3 | 1–5c then Cell 9 with `LEVELS=['fp32','int8']` | ~7h |
| 4 | 1–5c then Cell 9 with `LEVELS=['int4']` | ~4h |
| 5 | 1–5c then Cell 10 (Phase 3) + Cell 12 (figures) | ~4h |

**Every session ends with Cell 13** to persist results, and **every session from #2 starts by attaching your advc-results dataset** so Cell 5b + 5c restore prior work. Cells 8/9 auto-detect restored checkpoints and skip retraining.

### Interactive vs. Save & Run All
- **Interactive** (press ▶): recommended for the multi-session split — you control which levels run and call Cell 13 yourself.
- **Save & Run All (Commit)**: runs headless with your PC off, but re-runs the whole notebook from scratch on a fresh machine and **commits nothing if it times out**. Only use it for a scope that fits in 12h (e.g. one phase), and make sure Cell 5b restores prior checkpoints so it doesn't retrain finished levels.

### How to confirm it is NOT restarting from scratch
Run **Cell 5c** after restore. It prints, per level, whether a reusable `epoch07` checkpoint exists and which CSV rows are already recorded. In the training-cell output, a resumed level shows `... loading checkpoint ...` / `... checkpoint loaded.` with **no** `Epoch 1/7` lines; a resumed row shows `Resuming — N combination(s) already done`.